# Planners-13 — Mesurer la qualité des plans (benchmark de planificateurs)

[Version Python](Planners-13-Mesure-Qualite-Plans-Python.ipynb) · [Parcours de la partie](README.md)

**Navigation** : [Index](../../README.md) | [<< HTN](Planners-09-HTN-Python.ipynb) | [LLM Planning >>](../04-NeuroSymbolic/Planners-10-LLM-Planning-Python.ipynb)

***

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. **Mesurer** la qualité d'un plan (longueur, coût, nombre d'états explorés, temps)
2. **Comparer** des configurations de Fast Downward (heuristiques admissibles vs non admissibles, A* vs greedy) sur un **domaine non-trivial**
3. **Interpreter** pourquoi le choix de l'heuristique change radicalement l'efficience, à plan optimal égal
4. **Reproduire** une recherche naïve en C# et mesurer l'écart face à un vrai planificateur

### Prérequis

- [Planners-4](../02-Classical/Planners-04-Fast-Downward-Python.ipynb) (Fast Downward) et [Planners-5](../02-Classical/Planners-05-Heuristics-Python.ipynb) (Heuristiques)
- Fast Downward : Docker (image `jsboige/coursia-fast-downward`) **ou** le paquet Python `up-fast-downward`, installé par `requirements.txt`

### Durée estimée : 40 minutes

**Environnement du jumeau** — C# et Fast Downward, via le pilote du paquet up-fast-downward ou le service sur le port 8200.


***

## 1. Pourquoi mesurer la qualité des plans

Dans [Planners-4](../02-Classical/Planners-04-Fast-Downward-Python.ipynb), nous comparions les algorithmes et heuristiques de Fast Downward sur des problèmes **simples** (Blocks World à 3 blocs, Logistics, Ferry). La conclusion honnête de ce carnet était :

> *Pour un problème simple comme Blocks World, les différences sont minimes.*

C'est le piège de l'évaluation sur un cas **dégénéré** : quand le problème est trop facile, tous les planificateurs convergent vers le même résultat et on ne voit **pas** la capacité distinctive de l'outil. Un benchmark n'a de sens que si le problème est **assez riche** pour que le choix de la stratégie change un résultat **mesurable**.

Ce notebook reprend la méthode de Planners-4 (comparer des configurations Fast Downward) et l'applique à un domaine **non-trivial** — un Blocks World à 4 blocs — là où la discrimination devient nette : même plan optimal, mais un **écart net de plusieurs fois** sur le nombre d'états explorés.

## 2. Environnement

On réutilise l'infrastructure de Planners-4 : le conteneur Docker `jsboige/coursia-fast-downward` expose Fast Downward via une API HTTP sur le port 8200 ; sans Docker, le module `FastDownwardBackend.cs` de la série lance le même moteur par le pilote du paquet `up-fast-downward`. La cellule suivante vérifie sa disponibilité, et définit les deux fonctions qui porteront tout le benchmark.

In [1]:
#nullable enable
#load "../FastDownwardBackend.cs"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;
using System.Text.Json;
using System.Text.RegularExpressions;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// Le backend partagé démarre le vrai binaire via son pilote local ou utilise le service déjà lancé.
if (!FastDownwardBackend.Available)
    throw new InvalidOperationException("Installer up-fast-downward dans le Python accessible au noyau ou démarrer le service Fast Downward.");
Console.WriteLine($".NET {Environment.Version} | {FastDownwardBackend.Description}");

record SearchMeasurement(string Name, int Length, int Cost, int Evaluated, int Expanded, double Seconds, string[] Actions);

int SearchMetric(string stdout, string pattern, string label)
{
    var match = Regex.Match(stdout, pattern, RegexOptions.CultureInvariant);
    if (!match.Success)
        throw new FormatException($"Mesure Fast Downward absente : {label}");
    return int.Parse(match.Groups[1].Value, CultureInfo.InvariantCulture);
}

SearchMeasurement MeasurePlan(string domain, string problem, string name, string search, int timeout = 60)
{
    var timer = Stopwatch.StartNew();
    using var response = JsonDocument.Parse(FastDownwardBackend.PlanJson(domain, problem, search, timeout));
    double seconds = timer.Elapsed.TotalSeconds;
    var result = response.RootElement;
    int status = result.GetProperty("returncode").GetInt32();
    if (status != 0)
        throw new InvalidOperationException($"Fast Downward {name} a rendu le statut {status}.");
    // Les métriques d‘expansion et d'évaluation mesurent des étapes distinctes.
    string stdout = result.GetProperty("stdout").GetString() ?? "";
    var actions = Regex.Matches(stdout, @"^\s*((?:pick-up|put-down|stack|unstack)(?:\s+[a-z]+){1,2})\s+\(\d+\)\s*$",
        RegexOptions.Multiline | RegexOptions.CultureInvariant).Select(match => match.Groups[1].Value.Trim()).ToArray();
    return new(name, SearchMetric(stdout, @"Plan length:\s*(\d+)", "longueur"),
        SearchMetric(stdout, @"Plan cost:\s*(\d+)", "coût"), SearchMetric(stdout, @"Evaluated (\d+) state", "évalués"),
        SearchMetric(stdout, @"Expanded (\d+) state", "développés"), seconds, actions);
}

var searchConfigs = new (string Name, string Search)[]
{
    ("A* + blind", "astar(blind())"),
    ("A* + hmax", "astar(hmax())"),
    ("A* + lmcut", "astar(lmcut())"),
    ("A* + ff", "astar(ff())"),
    ("Greedy + ff", "lazy_greedy([ff()])")
};

The below script needs to be able to find the current output cell; this is an easy method to get it.

.NET 10.0.14 | Fast Downward 26.6, plugin local up_fast_downward


### Lecture du résultat

L'audit de l'environnement confirme les outils disponibles (voir le tableau imprimé ci-dessus) :

| Composant | Rôle |
|-----------|------|
| Docker | fait tourner le conteneur Fast Downward |
| Image `coursia-fast-downward` | embarque Fast Downward + API HTTP |
| Port 8200 | endpoint `/plan` (reçoit domaine + problème + config de recherche) |
| Paquet `up-fast-downward` | sans Docker, fournit le même moteur, lancé en local |

Le helper `MeasurePlan` encapsule l'appel au moteur (service HTTP ou pilote local) ; `SearchMetric` en extrait les mesures (longueur, coût, états évalués). Tout le benchmark repose sur ces deux fonctions.

***

## 3. Un domaine non-trivial

Nous reprenons le domaine STRIPS **Blocks World** de Planners-4, mais à une échelle où l'espace d'états devient non-négligeable : une **tour de 4 blocs** à construire. Le domaine est défini une fois pour toutes, puis un petit générateur de problème permet de changer la taille de la tour (ce que les exercices utiliseront).

In [2]:
#nullable enable
// Le même domaine STRIPS expose les quatre actions du monde des blocs.
const string BlocksDomain = """
(define (domain blocks)
  (:requirements :strips :typing)
  (:types block)
  (:predicates
    (on ?x - block ?y - block)
    (ontable ?x - block)
    (clear ?x - block)
    (handempty)
    (holding ?x - block)
  )
  (:action pick-up
    :parameters (?x - block)
    :precondition (and (clear ?x) (ontable ?x) (handempty))
    :effect (and (not (ontable ?x)) (not (clear ?x)) (not (handempty)) (holding ?x)))
  (:action put-down
    :parameters (?x - block)
    :precondition (holding ?x)
    :effect (and (not (holding ?x)) (clear ?x) (handempty) (ontable ?x)))
  (:action stack
    :parameters (?x - block ?y - block)
    :precondition (and (holding ?x) (clear ?y))
    :effect (and (not (holding ?x)) (not (clear ?y)) (clear ?x) (handempty) (on ?x ?y)))
  (:action unstack
    :parameters (?x - block ?y - block)
    :precondition (and (on ?x ?y) (clear ?x) (handempty))
    :effect (and (not (on ?x ?y)) (not (clear ?x)) (not (handempty)) (holding ?x) (clear ?y)))
)
""";
string TowerProblem(string[] blocks)
{
    var initial = blocks.Select(block => $"(ontable {block})")
        .Concat(blocks.Select(block => $"(clear {block})")).Append("(handempty)");
    var goals = blocks.Zip(blocks.Skip(1), (above, below) => $"(on {above} {below})");
    string goal = blocks.Length > 2 ? $"(and {string.Join(" ", goals)})" : goals.Single();
    return $$"""
    (define (problem tower-{{blocks.Length}})
      (:domain blocks)
      (:objects {{string.Join(" ", blocks)}} - block)
      (:init
        {{string.Join("\n    ", initial)}}
      )
      (:goal {{goal}})
    )
    """;
}

var blocks4 = new[] { "a", "b", "c", "d" };
var tower4 = TowerProblem(blocks4);
Console.WriteLine("Domaine : quatre actions STRIPS. But : a sur b, b sur c, c sur d.");
Console.WriteLine(tower4);

Domaine : quatre actions STRIPS. But : a sur b, b sur c, c sur d.


(define (problem tower-4)
  (:domain blocks)
  (:objects a b c d - block)
  (:init
    (ontable a)
    (ontable b)
    (ontable c)
    (ontable d)
    (clear a)
    (clear b)
    (clear c)
    (clear d)
    (handempty)
  )
  (:goal (and (on a b) (on b c) (on c d)))
)


### Lecture du résultat

Le problème affiché ci-dessus demande de construire une **tour de 4 blocs** (`a` au sommet, posé sur `b`, sur `c`, sur `d`). C'est le même domaine STRIPS que Planners-4, mais à une **échelle où l'espace d'états devient non-négligeable** : chaque bloc peut être sur la table, sur un autre bloc, ou en main, et l'ordre dans lequel on les empile génère de nombreux états intermédiaires.

C'est précisément cette richesse qui rend le benchmark informatif : sur 3 blocs, tous les planificateurs convergent vite ; sur 4 blocs, la **stratégie de recherche** commence à compter.

### Exercice 1 — Passer à 5 blocs

Construisez le problème **tower-5** (blocs `a b c d e`) avec `TowerProblem`, relancez le benchmark (les 5 configurations) et comparez le nombre d'états **évalués** de `A* + blind` à celui de `A* + lmcut`. Le ratio doit **croître** avec la taille de la tour : c'est la signature d'une heuristique qui « travaille ».

In [3]:
#nullable enable
// Exercice 1 : générer la tour de cinq blocs et relancer les cinq configurations.
var blocks5 = new[] { "a", "b", "c", "d", "e" };
var tower5 = TowerProblem(blocks5);
SearchMeasurement[] MeasureFive(string problem, (string Name, string Search)[] configurations)
{
    // TODO étudiant : appeler MeasurePlan pour chaque configuration et comparer les états.
    return Array.Empty<SearchMeasurement>();
}
var trial5 = MeasureFive(tower5, searchConfigs);
Console.WriteLine(trial5.Length == 0 ? "Exercice 1 à compléter." : $"{trial5.Length} configurations mesurées.");

Exercice 1 à compléter.


***

## 4. Le benchmark — 5 configurations, un même problème

On soumet le **même** problème à cinq configurations de Fast Downward : quatre recherches A* de qualité croissante (`blind`, `hmax`, `ff`, `lmcut`), dont trois heuristiques **admissibles** (`blind`, `hmax`, `lmcut`) qui garantissent chacune un plan optimal — `ff` ne l'est pas en général (plan relaxé : elle peut surestimer le coût restant), mais atteint ici le même optimum, ce qui rend le cas plus riche — et une recherche gloutonne (`lazy_greedy`), qui sacrifie toute garantie d'optimalité au profit de la vitesse. Pour chacune, on mesure la longueur du plan, son coût, le nombre d'états **évalués** et le temps.

In [4]:
#nullable enable
// Une configuration, un appel au solveur, un résultat au budget de soixante secondes.
var measurements = searchConfigs.Select(config => MeasurePlan(BlocksDomain, tower4, config.Name, config.Search)).ToArray();
Console.WriteLine("{0,-16} {1,8} {2,6} {3,9} {4,11} {5,8}",
    "Configuration", "Longueur", "Coût", "Évalués", "Développés", "Temps(s)");
foreach (var row in measurements)
{
    Console.WriteLine("{0,-16} {1,8} {2,6} {3,9} {4,11} {5,8:F3}",
        row.Name, row.Length, row.Cost, row.Evaluated, row.Expanded, row.Seconds);
}

Configuration    Longueur   Coût   Évalués  Développés Temps(s)


A* + blind              6      6       121          98    0.356


A* + hmax               6      6        56          27    0.137


A* + lmcut              6      6        22          10    0.149


A* + ff                 6      6        26          13    0.220


Greedy + ff            10     10        20          19    0.193


### Lecture du résultat — là où la discrimination apparaît

Le tableau ci-dessus est le cœur du notebook. Lisez-le en croisant deux colonnes :

- **Longueur / Coût** : les quatre recherches A* (`blind`, `hmax`, `ff`, `lmcut`) trouvent le **même plan optimal** — ces colonnes sont identiques d'une ligne à l'autre. Trois sont admissibles (`blind`, `hmax`, `lmcut`) : l'optimalité en découle. `ff` **n'est pas admissible** en général, mais atteint ici le même optimum — la garantie est une **conjonction** (« A* **et** heuristique admissible »), pas une propriété de la seule recherche.
- **Évalués** : c'est ici que tout se joue. La recherche **aveugle** (`blind`) explore un nombre d'états **très supérieur** à celui de `lmcut`, pour aboutir au **même** plan. L'heuristique ne sert pas à *améliorer* le plan (déjà optimal) ; elle sert à **l'atteindre en explorant moins**.

C'est la leçon que le cas à 3 blocs de Planners-4 masquait : *le choix de l'heuristique est un choix d'EFFICIENCE* (nombre d'états, temps), pas de qualité de plan, dès lors que l'heuristique est admissible. La ligne `Greedy + ff` (`lazy_greedy`) illustre le contrepoint : un plan **non optimal** — 10 actions contre 6 pour l'optimum, soit **4 actions de détour** — pour un nombre d'états évalués comparable à `lmcut` (20 contre 22) : sur cette instance, la vitesse n'est pas acquise, seule la garantie est perdue.

**Indice :** le but de la tour à 5 blocs est `(and (on a b) (on b c) (on c d) (on d e))`. La longueur optimale est de 8 actions. Ce qui change, c'est le nombre d'états que chaque configuration doit évaluer pour l'atteindre : observez l'écart `blind`/`lmcut`.

### Exercice 2 — Une heuristique admissible en mode glouton

Ajoutez la configuration `greedy + lmcut` (recherche gloutonne avec une heuristique **admissible**) et observez si le plan trouvé reste **optimal en longueur**.

La recherche gloutonne évalue `f(n) = h(n)` **sans** le coût accumulé `g(n)` : l'admissibilité de `h` ne suffit plus à garantir l'optimalité.

In [5]:
#nullable enable
// Exercice 2 : conserver la même tour, remplacer la stratégie de recherche pour LM-cut.
SearchMeasurement? CompareGreedyLmcut(string problem)
{
    // TODO étudiant : tester lazy_greedy([lmcut()]) et comparer la longueur obtenue.
    return null;
}
var greedyLmcut = CompareGreedyLmcut(tower4);
Console.WriteLine(greedyLmcut is null ? "Exercice 2 à compléter."
    : $"Glouton {greedyLmcut.Length} actions | A* {measurements.Single(row => row.Name == "A* + lmcut").Length} actions");

Exercice 2 à compléter.


***

## 5. Un vrai planificateur vs une recherche naïve

Pour isoler la valeur de l'heuristique, la cellule suivante reproduit **en C#, avec un parcours BFS indépendant**, une recherche **aveugle** (BFS) du même problème. C'est le contrepoint le plus pur : même problème, même complétude, mais **aucune heuristique** pour guider l'exploration. On compare alors le nombre d'états que cette recherche naïve explore à celui de `A* + lmcut`.

In [6]:
#nullable enable
record BlocksState(string[] Support, string? Held)
{
    public string Key => $"{string.Join("|", Support)}#{Held}";
}

BlocksState InitialBlocks(string[] blocks) => new(blocks.Select(_ => "table").ToArray(), null);

bool GoalReached(BlocksState state, string[] blocks)
{
    // L'ordre des blocs nomme les relations on demandées dans le problème PDDL.
    return blocks.Skip(1).Select((below, i) => state.Support[i] == below).All(match => match);
}

IEnumerable<(BlocksState State, string Action)> Successors(BlocksState state, string[] blocks)
{
    bool Clear(string name) => !state.Support.Contains(name);
    BlocksState Move(int index, string target, string? held)
    {
        var support = state.Support.ToArray();
        support[index] = target;
        return new(support, held);
    }
    for (int i = 0; i < blocks.Length; i++)
    {
        string block = blocks[i];
        string support = state.Support[i];
        // La main vide permet de prendre un bloc dégagé, sur table ou sur un autre bloc.
        if (state.Held is null && support == "table" && Clear(block))
            yield return (Move(i, "hand", block), $"pick-up {block}");
        if (state.Held == block)
        {
            yield return (Move(i, "table", null), $"put-down {block}");
            foreach (string target in blocks.Where(x => x != block && Clear(x)))
                yield return (Move(i, target, null), $"stack {block} {target}");
        }
        if (state.Held is null && support != "table" && support != "hand" && Clear(block))
            yield return (Move(i, "hand", block), $"unstack {block} {support}");
    }
}

(int? Length, int Expanded) BreadthFirst(string[] blocks)
{
    var root = InitialBlocks(blocks);
    if (GoalReached(root, blocks)) return (0, 0);
    var queue = new Queue<(BlocksState State, int Depth)>();
    queue.Enqueue((root, 0));
    var seen = new HashSet<string> { root.Key };
    int expanded = 0;
    while (queue.TryDequeue(out var node))
    {
        expanded++;
        foreach (var (successor, _) in Successors(node.State, blocks))
        {
            if (!seen.Add(successor.Key)) continue;
            if (GoalReached(successor, blocks)) return (node.Depth + 1, expanded);
            queue.Enqueue((successor, node.Depth + 1));
        }
    }
    return (null, expanded);
}

bool ValidatePlan(SearchMeasurement plan, string[] blocks)
{
    // Rejouer les actions avec un moteur indépendant valide le plan plutôt que sa longueur seule.
    if (plan.Actions.Length != plan.Length) return false;
    var state = InitialBlocks(blocks);
    foreach (string action in plan.Actions)
    {
        var step = Successors(state, blocks).SingleOrDefault(item => item.Action == action);
        if (step.State is null) return false;
        state = step.State;
    }
    return GoalReached(state, blocks);
}

var bfs = BreadthFirst(blocks4);
var lmcut = measurements.Single(row => row.Name == "A* + lmcut");
Console.WriteLine($"BFS sans heuristique : longueur {bfs.Length}, {bfs.Expanded} états développés.");
Console.WriteLine($"A* + lmcut : longueur {lmcut.Length}, {lmcut.Evaluated} états évalués.");
Console.WriteLine($"Rapport indicatif des compteurs BFS développés / A* évalués : {bfs.Expanded / (double)lmcut.Evaluated:F2}.");
foreach (var row in measurements)
    Console.WriteLine($"{row.Name,-16} | plan rejoué et vérifié : {ValidatePlan(row, blocks4)}");

BFS sans heuristique : longueur 6, 95 états développés.


A* + lmcut : longueur 6, 22 états évalués.


Rapport indicatif des compteurs BFS développés / A* évalués : 4.32.


A* + blind       | plan rejoué et vérifié : True


A* + hmax        | plan rejoué et vérifié : True


A* + lmcut       | plan rejoué et vérifié : True


A* + ff          | plan rejoué et vérifié : True


Greedy + ff      | plan rejoué et vérifié : True


### Lecture du résultat — un vrai planificateur vs une recherche naïve

La cellule précédente reproduit, en une quarantaine de lignes de C#, une recherche **aveugle** (BFS) du même problème, sans aucune heuristique. Le message est le même que pour Fast Downward : la recherche naïve explore **beaucoup** d'états, là où `A* + lmcut` s'en tient à une fraction.

L'écart affiché (le ratio d'états explorés) est la traduction chiffrée d'une idée simple : **une heuristique admissible est ce qui sépare un planificateur qui « devine bien » d'une recherche qui « essaie tout »**. C'est la valeur distinctive de l'outil SOTA, mesurable — et c'est exactement ce qu'un benchmark sur un problème trop simple ne permet pas de voir.

**Indice :** comparez la colonne *Longueur* de `Greedy + lmcut` à celle de `A* + lmcut`. S'ils diffèrent, c'est la preuve que l'admissibilité garantit l'optimalité **pour A***, pas pour le glouton.

### Exercice 3 — Reconstruire la séquence du plan

Modifiez `BreadthFirst` pour qu'il renvoie aussi la **séquence d'actions** du plan (et pas seulement sa longueur), puis vérifiez que cette séquence satisfait bien le but.

C'est le passage de la *mesure* (combien d'états) à la *preuve* (quel plan) : un benchmark doit pouvoir produire le plan qu'il compte.

In [7]:
#nullable enable
// Exercice 3 : les parents et actions sont enregistrés au premier passage par un état.
string[] ReconstructBreadthFirstPlan(string[] blocks)
{
    // TODO étudiant : conserver les parents et remonter depuis le premier état but.
    return Array.Empty<string>();
}
var trialPlan = ReconstructBreadthFirstPlan(blocks4);
Console.WriteLine(trialPlan.Length == 0 ? "Exercice 3 à compléter."
    : $"Actions du BFS : {string.Join(" → ", trialPlan)}");

Exercice 3 à compléter.


## Vérifiez votre compréhension

1. Que conclure si la recherche aveugle et LM-cut rendent toutes deux six actions ?
2. Quel compromis mesure le plan glouton de dix actions ?
3. Qu'apporte la relecture du plan par le moteur de transitions C# ?

<details>
<summary>Éléments de réponse</summary>

1. Les plans ont une longueur identique, mais Fast Downward évalue 121 états avec la recherche aveugle et 22 avec LM-cut. La différence porte sur le travail de recherche.
2. Le glouton évalue 20 états sur ce cas et rend un coût de 10 contre 6 pour le plan optimal.
3. Elle vérifie la légalité de chaque transition et l'atteinte de la tour, en plus de la longueur annoncée par le solveur.

</details>

**Indice :** la remontée de `parent` donne la chaîne d'états ; pour en extraire l'action, comparez `on` et `holding` entre un état et son parent (un seul bloc change de support, et `holding` passe de `None` à un bloc ou vice-versa).

***

## À retenir

1. **Un benchmark n'a de sens que sur un problème non-trivial** : sur un cas dégénéré, tous les planificateurs convergent et on ne mesure rien.
2. **Admissible = plan optimal garanti** : `blind`, `hmax`, `lmcut` donnent la même longueur de plan ; l'admissibilité protège la qualité.
3. **L'heuristique est un choix d'efficience** : à plan optimal égal, `lmcut` explore nettement moins d'états que `blind`. C'est le rôle de l'heuristique.
4. **Satisficing (greedy) = vitesse sans garantie** : plus rapide à lancer, plan non garanti optimal.
5. **La valeur d'un vrai planificateur se mesure** : face à une recherche naïve (BFS), l'écart d'états explorés est l'indice de sa capacité distinctive.

## Pour aller plus loin

- Pattern databases et heuristiques additives (`merge-and-shrink`) : au-delà de `lmcut`.
- Domaines à coûts explicites (`:metric`) : quand la longueur ne suffit plus.
- [Planners-09-HTN-Python](Planners-09-HTN-Python.ipynb) : la planification hiérarchique, un autre levier d'efficience.

***

**Notebook précédent** : [Planners-09-HTN-Python](Planners-09-HTN-Python.ipynb) · **Suivant** : [Planners-10-LLM-Planning-Python](../04-NeuroSymbolic/Planners-10-LLM-Planning-Python.ipynb)